# Barcelona Attendee Occupancy Stream

Generates changing occupancy signals for CCIB and 14 attendee-relevant hotels, cafes, and interesting spots across Barcelona. Before streaming, each source footprint is converted into a literal GeoJSON silhouette centered on the venue: a stepped hotel with windows, a cafe cup with handle, a wide conference building, or a hollow magnifying glass.

**Event window:** Sun 27 Sep – Thu 1 Oct 2026 (Microsoft Fabric Conference, held at **CCIB**, Sant Martí district) — the same fixed event window and Europe/Madrid (CEST, UTC+2) time base used by the Barcelona traffic generator. The full dataset is pre-generated at a 5-minute cadence across the event window, then streamed chronologically at compressed playback speed (`SPEED_FACTOR`).

Every five-minute Barcelona-time window applies a synthetic spike or drop to exactly one rotating venue so anomaly detection can be demonstrated. The Eventhouse contract includes `occupancySignalId`, `category`, `currentOccupancy`, `totalOccupancy`, `geometry`, and the Barcelona-local `timestamp`. Capacities, occupancy values, anomalies, and symbol dimensions are demo estimates.


In [1]:
# Install the Azure Event Hubs SDK
%pip install azure-eventhub --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Fabric Eventstream / Azure Event Hub connection ───────────────────────────
# Paste from: Fabric portal -> Eventstream -> Sources -> Custom endpoint
OCCUPANCY_EH_CONN_STR = 'OCCUPANCY_EVENTHUB_CONNECTION_STRING_PLACEHOLDER'
OCCUPANCY_EH_NAME     = 'OCCUPANCY_EVENTHUB_NAME_PLACEHOLDER'

In [ ]:
import json
import math
import random
import time
from datetime import datetime, timedelta, timezone, date
from pathlib import Path
import pandas as pd

In [ ]:
GEOJSON_PATH = Path('/lakehouse/default/Files/occupancy_locations.geojson')

# ── Event window (Europe/Madrid, CEST = UTC+2 in late Sep / early Oct) ───────
BCN_TZ      = timezone(timedelta(hours=2))
EVENT_START = date(2026, 9, 27)   # Sun - Partner Day / registration
EVENT_END   = date(2026, 10, 1)   # Thu - final day / departures

# ── Data cadence & playback compression for streaming ─────────────────────────
INTERVAL_MINUTES   = 5                                        # data granularity (matches the anomaly window)
SPEED_FACTOR       = 12                                       # 12x -> 5 min real = 25 s playback
EMIT_DELAY_SECONDS = (INTERVAL_MINUTES * 60) / SPEED_FACTOR

ANOMALY_WINDOW_MINUTES = INTERVAL_MINUTES
ANOMALY_SPIKE_RATIO = 0.96
ANOMALY_DROP_RATIO = 0.04
RANDOM_SEED = 42
EXPECTED_LOCATION_COUNT = 15
EXPECTED_COLUMNS = ['occupancySignalId', 'category', 'currentOccupancy', 'totalOccupancy', 'geometry', 'timestamp']
ALLOWED_CATEGORIES = {'conference_venue', 'hotel', 'cafe', 'interesting_spot'}

random.seed(RANDOM_SEED)
print(f'Event window: {EVENT_START} -> {EVENT_END} | interval={INTERVAL_MINUTES}min | '
      f'playback speed={SPEED_FACTOR}x (~{EMIT_DELAY_SECONDS:.1f}s per batch)')

In [5]:
def category_shape(feature):
    source_ring = feature['geometry']['coordinates'][0]
    longitudes = [point[0] for point in source_ring]
    latitudes = [point[1] for point in source_ring]
    center_lng = (min(longitudes) + max(longitudes)) / 2
    center_lat = (min(latitudes) + max(latitudes)) / 2
    dx, dy = 0.00055, 0.00042

    def point(x, y):
        return [round(center_lng + x * dx, 6), round(center_lat + y * dy, 6)]

    category = feature['properties']['category']
    if category == 'hotel':
        outer = [
            point(-0.75, -1.00), point(0.75, -1.00), point(0.75, 0.65),
            point(0.42, 0.65), point(0.42, 1.00), point(-0.42, 1.00),
            point(-0.42, 0.65), point(-0.75, 0.65), point(-0.75, -1.00),
        ]
        holes = [
            [point(-0.45, -0.55), point(-0.15, -0.55), point(-0.15, -0.15), point(-0.45, -0.15), point(-0.45, -0.55)],
            [point(0.15, -0.55), point(0.45, -0.55), point(0.45, -0.15), point(0.15, -0.15), point(0.15, -0.55)],
            [point(-0.45, 0.10), point(-0.15, 0.10), point(-0.15, 0.45), point(-0.45, 0.45), point(-0.45, 0.10)],
            [point(0.15, 0.10), point(0.45, 0.10), point(0.45, 0.45), point(0.15, 0.45), point(0.15, 0.10)],
        ]
        return {'type': 'Polygon', 'coordinates': [outer, *holes]}

    if category == 'cafe':
        outer = [
            point(-0.85, 0.55), point(0.35, 0.55), point(0.35, 0.38),
            point(0.78, 0.38), point(0.95, 0.20), point(0.95, -0.15),
            point(0.78, -0.35), point(0.42, -0.35), point(0.28, -0.75),
            point(0.05, -0.95), point(-0.50, -0.95), point(-0.72, -0.75),
            point(-0.85, 0.55),
        ]
        handle_hole = [
            point(0.42, 0.20), point(0.70, 0.20), point(0.78, 0.10),
            point(0.78, -0.08), point(0.68, -0.18), point(0.42, -0.18),
            point(0.42, 0.20),
        ]
        return {'type': 'Polygon', 'coordinates': [outer, handle_hole]}

    if category == 'conference_venue':
        outer = [
            point(-1.35, -0.90), point(-0.22, -0.90), point(-0.22, -0.35),
            point(0.22, -0.35), point(0.22, -0.90), point(1.35, -0.90),
            point(1.35, 0.35),
            point(0.95, 0.35), point(0.95, 0.62), point(0.45, 0.62),
            point(0.45, 0.88), point(-0.45, 0.88), point(-0.45, 0.62),
            point(-0.95, 0.62), point(-0.95, 0.35), point(-1.35, 0.35),
            point(-1.35, -0.90),
        ]
        return {'type': 'Polygon', 'coordinates': [outer]}

    outer = [
        point(-0.82, 0.28), point(-0.62, 0.72), point(-0.22, 0.98),
        point(0.28, 0.98), point(0.70, 0.70), point(0.88, 0.28),
        point(0.78, -0.18), point(0.50, -0.48), point(1.10, -0.98),
        point(0.78, -1.25), point(0.18, -0.66), point(-0.25, -0.62),
        point(-0.65, -0.36), point(-0.82, 0.28),
    ]
    lens_hole = [
        point(-0.45, 0.25), point(-0.30, 0.58), point(0.00, 0.75),
        point(0.35, 0.62), point(0.55, 0.30), point(0.45, -0.05),
        point(0.15, -0.28), point(-0.20, -0.22), point(-0.43, 0.02),
        point(-0.45, 0.25),
    ]
    return {'type': 'Polygon', 'coordinates': [outer, lens_hole]}


with GEOJSON_PATH.open(encoding='utf-8') as geojson_file:
    location_collection = json.load(geojson_file)

locations = location_collection['features']
for feature in locations:
    feature['geometry'] = category_shape(feature)

location_ids = [feature['properties']['occupancySignalId'] for feature in locations]
assert location_collection['type'] == 'FeatureCollection'
assert len(locations) == EXPECTED_LOCATION_COUNT
assert len(set(location_ids)) == EXPECTED_LOCATION_COUNT
assert all(feature['properties']['category'] in ALLOWED_CATEGORIES for feature in locations)
assert all(feature['properties']['totalOccupancy'] > 0 for feature in locations)
assert all(feature['geometry']['type'] == 'Polygon' for feature in locations)
assert all(ring[0] == ring[-1] for feature in locations for ring in feature['geometry']['coordinates'])
assert all(len(feature['geometry']['coordinates'][0]) > 5 for feature in locations)

catalog = pd.DataFrame([
    {**feature['properties'], 'geometry': feature['geometry']}
    for feature in locations
])
print(f'Loaded {len(catalog)} occupancy locations with category-shaped polygons')
display(catalog[['occupancySignalId', 'name', 'category', 'totalOccupancy']])

Loaded 15 occupancy locations with category-shaped polygons


In [ ]:
PEAK_WINDOWS = (
    ('morning', 8 * 60, 9 * 60 + 30),
    ('midday', 13 * 60, 15 * 60),
    ('afternoon', 17 * 60, 19 * 60 + 30),
)

PEAK_OCCUPANCY_RATIOS = {
    'conference_venue': {'morning': 0.92, 'midday': 0.86, 'afternoon': 0.78},
    'hotel': {'morning': 0.70, 'midday': 0.62, 'afternoon': 0.74},
    'cafe': {'morning': 0.82, 'midday': 0.90, 'afternoon': 0.76},
    'interesting_spot': {'morning': 0.46, 'midday': 0.68, 'afternoon': 0.72},
}

OFF_PEAK_OCCUPANCY_RATIOS = {
    'conference_venue': 0.18,
    'hotel': 0.58,
    'cafe': 0.12,
    'interesting_spot': 0.16,
}

def peak_window_factor(local_time):
    """Return the active peak name and a smooth 0.75-1.0 factor, or (None, 0.0) off peak."""
    minute_of_day = local_time.hour * 60 + local_time.minute
    for peak_name, start_minute, end_minute in PEAK_WINDOWS:
        if start_minute <= minute_of_day < end_minute:
            progress = (minute_of_day - start_minute) / (end_minute - start_minute)
            return peak_name, 0.75 + 0.25 * math.sin(math.pi * progress)
    return None, 0.0

def target_occupancy_ratio(category, local_time):
    peak_name, peak_factor = peak_window_factor(local_time)
    off_peak_ratio = OFF_PEAK_OCCUPANCY_RATIOS[category]
    if not peak_name:
        if category == 'hotel' and (local_time.hour < 8 or local_time.hour >= 19):
            return 0.84
        return off_peak_ratio

    peak_ratio = PEAK_OCCUPANCY_RATIOS[category][peak_name]
    return min(0.96, off_peak_ratio + (peak_ratio - off_peak_ratio) * peak_factor)

In [ ]:
def active_anomaly(local_time):
    window_seconds = ANOMALY_WINDOW_MINUTES * 60
    window_number = int(local_time.timestamp() // window_seconds)
    feature = locations[window_number % len(locations)]
    anomaly_kind = 'spike' if window_number % 2 == 0 else 'drop'
    anomaly_ratio = ANOMALY_SPIKE_RATIO if anomaly_kind == 'spike' else ANOMALY_DROP_RATIO
    return feature['properties']['occupancySignalId'], anomaly_kind, anomaly_ratio


def build_occupancy_batch(local_time):
    anomaly_id, _, anomaly_ratio = active_anomaly(local_time)
    records = []

    for feature in locations:
        props = feature['properties']
        signal_id = props['occupancySignalId']
        capacity = props['totalOccupancy']
        target = capacity * target_occupancy_ratio(props['category'], local_time)
        previous = occupancy_state[signal_id]
        noise = random.gauss(0, max(1, capacity * 0.012))
        current = round(previous + 0.22 * (target - previous) + noise)

        if signal_id == anomaly_id:
            anomaly_noise = random.gauss(0, capacity * 0.008)
            current = round(capacity * anomaly_ratio + anomaly_noise)

        current = max(0, min(capacity, current))
        occupancy_state[signal_id] = current
        records.append({
            'occupancySignalId': signal_id,
            'category': props['category'],
            'currentOccupancy': current,
            'totalOccupancy': capacity,
            'geometry': feature['geometry'],
            'timestamp': local_time.isoformat(),
        })

    return records

print('Occupancy dynamics functions defined.')

In [ ]:
# ── Pre-generate the full event-window dataset (same approach as the traffic generator) ──
timestamps = pd.date_range(
    start=datetime(EVENT_START.year, EVENT_START.month, EVENT_START.day, tzinfo=BCN_TZ),
    end=datetime(EVENT_END.year, EVENT_END.month, EVENT_END.day, 23, 55, tzinfo=BCN_TZ),
    freq=f'{INTERVAL_MINUTES}min',
)

# Seed each venue's occupancy state at the target ratio for the first timestamp of the event window
occupancy_state = {}
for feature in locations:
    props = feature['properties']
    initial_ratio = target_occupancy_ratio(props['category'], timestamps[0])
    occupancy_state[props['occupancySignalId']] = round(props['totalOccupancy'] * initial_ratio)

all_records = []
anomaly_by_ts = {}
for ts in timestamps:
    anomaly_id, anomaly_kind, _ = active_anomaly(ts)
    anomaly_by_ts[ts.isoformat()] = (anomaly_id, anomaly_kind)
    all_records.extend(build_occupancy_batch(ts))

df_occupancy = pd.DataFrame(all_records, columns=EXPECTED_COLUMNS)
print(f'Generated {len(df_occupancy):,} records | {len(locations)} locations | '
      f'{timestamps.min()} -> {timestamps.max()}')
df_occupancy.head(10)

In [ ]:
# Sanity check: schema, occupancy bounds, and anomaly rotation across the event window
assert df_occupancy.columns.tolist() == EXPECTED_COLUMNS
assert (df_occupancy['currentOccupancy'].between(0, df_occupancy['totalOccupancy'])).all()
assert df_occupancy.groupby('timestamp')['occupancySignalId'].apply(lambda ids: ids.is_unique).all()

sample_ts = timestamps[100]
sample_anomaly_id, sample_anomaly_kind = anomaly_by_ts[sample_ts.isoformat()]
print(f'Active 5-minute anomaly at {sample_ts.isoformat()}: {sample_anomaly_kind} at {sample_anomaly_id}')

print('\nFirst anomaly rotation (id, kind):')
for ts in timestamps[:len(locations) * 2]:
    anomaly_id, anomaly_kind = anomaly_by_ts[ts.isoformat()]
    print(f'  {ts.strftime("%a %d %H:%M")} -> {anomaly_kind:>5} @ {anomaly_id}')

## Category shapes and Eventstream

The streamed `geometry` contains the symbol itself: `hotel` is a stepped hotel with window holes, `cafe` is a cup with a hollow handle, `conference_venue` is a wide conference building with an entrance notch, and `interesting_spot` is a magnifying glass with a hollow lens. Render `geometry` as a filled GeoJSON polygon layer; no separate map icon configuration is required.

Every five-minute Barcelona-time window (within the fixed `EVENT_START`–`EVENT_END` conference window) selects exactly one venue for a synthetic anomaly. Even-numbered windows spike near 96% occupancy and odd-numbered windows drop near 4%; small noise keeps the signal realistic. The anomaly is visible through `currentOccupancy`, and each emitted message includes a Barcelona-local ISO 8601 `timestamp`.

The cell below streams every 5-minute window (all 15 locations) as one Event Hub batch, in chronological order, at compressed playback speed (`SPEED_FACTOR`) — exactly like the traffic generator. Set `LIVE_SLEEP = False` to bulk-send the whole dataset instantly (useful for backfill/testing). Requires `pip install azure-eventhub` and a valid connection string/name in the config cell above.


In [ ]:
from azure.eventhub import EventData, EventHubProducerClient

if not OCCUPANCY_EH_CONN_STR or not OCCUPANCY_EH_NAME:
    raise ValueError('Set OCCUPANCY_EVENTHUB_CONNECTION_STRING and OCCUPANCY_EVENTHUB_NAME before streaming.')

STREAM_COLUMNS = EXPECTED_COLUMNS
LIVE_SLEEP = True   # False = bulk send without delay

producer = EventHubProducerClient.from_connection_string(
    conn_str=OCCUPANCY_EH_CONN_STR, eventhub_name=OCCUPANCY_EH_NAME,
)

total_sent = 0
with producer:
    for ts, grp in df_occupancy.groupby('timestamp', sort=True):
        batch = producer.create_batch()
        for rec in grp[STREAM_COLUMNS].to_dict(orient='records'):
            batch.add(EventData(json.dumps(rec, default=str)))
        producer.send_batch(batch)
        total_sent += len(grp)
        anomaly_id, anomaly_kind = anomaly_by_ts[ts]
        print(f'[{datetime.now().strftime("%H:%M:%S")}] ts={ts}  sent={len(grp):2d}  anomaly={anomaly_kind}:{anomaly_id}')
        if LIVE_SLEEP:
            time.sleep(EMIT_DELAY_SECONDS)

print(f'\nDone. Total records sent to Eventstream: {total_sent:,}')